# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) Two paper findings and my methodology questions

**Finding A:** ___ (one sentence, in my own words, from the paper)
- **Question I would ask:** Where does the label for this finding come from, and is it measured after the window the features come from?
- **Why it matters:** If the label overlaps the feature window, the result could be partly built in.
- **Why I think the authors made this choice:** ___ (for example, a broad audience, or disclosed limits)

**Finding B:** ___ (one sentence, in my own words)
- **Question I would ask:** Does the validation design (split by ___) support the claim for pages or clients the model hasn't seen?
- **Why it matters:** A random split can reward memorizing one client's pages.
- **What I'd like to see next:** ___ (for example, results by client or by month)

I'm asking these as practice at the next level of rigor, the way I'd want my own work reviewed. I'm not grading the paper.

In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score


## 2. My Model Under an Honest Split

I compared a random row split with a grouped split by client.

| Metric | Random Split | Grouped Split |
|---|---:|---:|
| Precision@50 | 0.740 | 0.640 |
| AUC | 0.614 | 0.593 |

Both metrics decreased under grouped validation. This suggests that random splitting may provide an optimistic estimate of model performance when records from the same clients appear in both training and testing sets.

The grouped split contained 15 training clients and 7 testing clients, with zero overlapping clients.

However, one client represented 96.6% of the 1,400 test rows. This severe imbalance limits how confidently these results can be generalized to other clients.

In [ ]:

import duckdb
import numpy as np
import pandas as pd

from huggingface_hub import hf_hub_download
from google.colab import userdata

TOKEN = userdata.get("HF_TOKEN")
REPO = "FlyRank/internship-warehouse"

FACT = hf_hub_download(
    REPO,
    "fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=TOKEN
)

DIM_CONTENT = hf_hub_download(
    REPO,
    "dim_content.parquet",
    repo_type="dataset",
    token=TOKEN
)

con = duckdb.connect()


In [ ]:

df = con.sql(f"""
WITH early AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_early,
        SUM(gsc_clicks) AS clicks_early,
        AVG(gsc_avg_position) AS avg_position_early,

        SUM(gsc_impressions) FILTER (
            WHERE report_date <= DATE '2026-03-10'
        ) AS imp_h1,

        SUM(gsc_impressions) FILTER (
            WHERE report_date > DATE '2026-03-10'
        ) AS imp_h2

    FROM read_parquet('{FACT}')
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
    GROUP BY 1, 2
),

late AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_late

    FROM read_parquet('{FACT}')
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-21'
                          AND DATE '2026-03-31'
    GROUP BY 1, 2
)

SELECT
    e.client_hash_id,
    e.content_hash_id,
    e.impressions_early,

    e.clicks_early * 1.0 /
        NULLIF(e.impressions_early, 0) AS ctr_early,

    e.avg_position_early,

    (COALESCE(e.imp_h2, 0) + 1.0) /
        (COALESCE(e.imp_h1, 0) + 1.0) AS impr_trend_early,

    DATE '2026-03-20' - c.content_updated_date
        AS days_since_update,

    c.search_volume,

    (COALESCE(l.clicks_late, 0) / 11.0)
        < 0.8 * (e.clicks_early / 20.0) AS label

FROM early e

JOIN read_parquet('{DIM_CONTENT}') c
    USING (client_hash_id, content_hash_id)

LEFT JOIN late l
    USING (client_hash_id, content_hash_id)

WHERE e.clicks_early >= 5
  AND c.content_updated_date <= DATE '2026-03-20'

""").df()

df["label"] = df["label"].astype(int)
df["days_since_update"] = df["days_since_update"].astype(float)
df["search_volume"] = df["search_volume"].astype(float)

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (2526, 9)


,client_hash_id,content_hash_id,impressions_early,ctr_early,avg_position_early,impr_trend_early,days_since_update,search_volume,label
0,client_73cda7b4e4f265ea,content_e69b442038b895f0,11532.0,0.000780,3.490073,0.737048,23.0,0.0,1
1,client_73cda7b4e4f265ea,content_0d6905b49fa1729c,23024.0,0.000912,3.924928,1.159430,23.0,0.0,0
2,client_73cda7b4e4f265ea,content_e9f2d0579387d3c3,42886.0,0.001096,5.008406,0.853174,23.0,0.0,1
3,client_73cda7b4e4f265ea,content_0319c9c35522464a,8324.0,0.001081,3.091726,0.696760,23.0,0.0,0
4,client_73cda7b4e4f265ea,content_cb0d950177839034,3423.0,0.002337,4.141979,1.030231,23.0,0.0,1


In [ ]:

df["search_volume"] = df["search_volume"].fillna(
    df["search_volume"].median()
)

df["impressions_log"] = np.log1p(df["impressions_early"])
df["search_volume_log"] = np.log1p(df["search_volume"])

df["impr_trend_log"] = np.log(
    df["impr_trend_early"].clip(0.05, 20)
)

print(df.shape)


(2526, 12)


In [ ]:

FEATS = [
    "impressions_log",
    "ctr_early",
    "avg_position_early",
    "days_since_update",
    "search_volume_log",
    "impr_trend_log"
]

LABEL = "label"
GROUP = "client_hash_id"

# Keep rows even if some features are missing.
d = df.dropna(subset=[LABEL, GROUP]).copy()
d = d.reset_index(drop=True)

print("d shape:", d.shape)
print("Missing feature values:")
print(d[FEATS].isna().sum())


d shape: (2526, 12)
Missing feature values:
impressions_log       0
ctr_early             0
avg_position_early    0
days_since_update     0
search_volume_log     0
impr_trend_log        0
dtype: int64


In [ ]:

def precision_at_k(y, scores, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y)[order].mean())


def fit_eval(train, test):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=20,
            random_state=0,
            n_jobs=-1
        )
    )

    model.fit(train[FEATS], train[LABEL])

    scores = model.predict_proba(test[FEATS])[:, 1]

    precision = precision_at_k(test[LABEL], scores, 50)
    auc = roc_auc_score(test[LABEL], scores)

    return round(precision, 3), round(auc, 3)


print("Functions are ready!")


Functions are ready!


In [ ]:

# BEFORE: Random row split
tr_a, te_a = train_test_split(
    d,
    test_size=0.3,
    random_state=0
)

before = fit_eval(tr_a, te_a)

# AFTER: Grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.3,
    random_state=0
)

tr_i, te_i = next(
    gss.split(d, groups=d[GROUP])
)

tr_g = d.iloc[tr_i]
te_g = d.iloc[te_i]

after = fit_eval(tr_g, te_g)

comparison = pd.DataFrame({
    "Split": ["Random rows (before)", "Grouped by client (after)"],
    "Precision@50": [before[0], after[0]],
    "AUC": [before[1], after[1]]
})

display(comparison)

print("Random split:", before)
print("Grouped split:", after)
print(
    "Group overlap:",
    len(set(tr_g[GROUP]) & set(te_g[GROUP]))
)


,Split,Precision@50,AUC
0,Random rows (before),0.74,0.614
1,Grouped by client (after),0.64,0.593


Random split: (0.74, np.float64(0.614))
Grouped split: (0.64, np.float64(0.593))
Group overlap: 0


### Feature Dependency and Leakage Audit

I examined the individual relationship between each feature and the target using single-feature AUC.

The strongest individual signals were:

- `avg_position_early`: 0.591
- `impr_trend_log`: 0.412
- `impressions_log`: 0.430

The `ctr_early` feature had an AUC of 0.498, showing almost no individual ranking ability in this dataset.

I also compared the grouped model with and without `ctr_early`, using the same train/test split.

| Model | Precision@50 | AUC |
|---|---:|---:|
| All features | 0.640 | 0.593 |
| Without ctr_early | 0.680 | 0.600 |

Removing `ctr_early` improved both metrics in this experiment.

This suggests that `ctr_early` did not add useful predictive information under this particular split. Its shared dependency with the target formula deserves investigation, but this comparison alone does not prove leakage.

In [ ]:

# Single-feature AUC
rows = []

for feature in FEATS:
    auc = roc_auc_score(d[LABEL], d[feature])

    # AUC below 0.5 can indicate an inverse relationship
    strength = max(auc, 1 - auc)

    rows.append({
        "feature": feature,
        "single_feature_AUC": round(auc, 3),
        "AUC_strength": round(strength, 3)
    })

single_feature_results = pd.DataFrame(rows).sort_values(
    "AUC_strength",
    ascending=False
)

display(single_feature_results)


,feature,single_feature_AUC,AUC_strength
2,avg_position_early,0.591,0.591
5,impr_trend_log,0.412,0.588
0,impressions_log,0.430,0.570
4,search_volume_log,0.453,0.547
3,days_since_update,0.508,0.508
1,ctr_early,0.498,0.502


In [ ]:

correlations = d[FEATS + [LABEL]].corr(
    numeric_only=True
)[LABEL].drop(LABEL)

display(
    correlations.sort_values(
        ascending=False
    ).round(3)
)


,label
avg_position_early,0.110
days_since_update,0.044
ctr_early,0.010
search_volume_log,-0.078
impr_trend_log,-0.118
impressions_log,-0.125


In [ ]:

train_clients = set(tr_g[GROUP])
test_clients = set(te_g[GROUP])

overlap = train_clients.intersection(test_clients)

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Overlapping clients:", len(overlap))
print("Overlap:", overlap)


Training clients: 15
Testing clients: 7
Overlapping clients: 0
Overlap: set()


In [ ]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=20,
        random_state=0,
        n_jobs=-1
    )
)

model.fit(tr_g[FEATS], tr_g[LABEL])

te = te_g.copy()
te["score"] = model.predict_proba(te[FEATS])[:, 1]

# Top 50 predictions
top = te.sort_values(
    "score",
    ascending=False
).head(50)

false_positives = top[top[LABEL] == 0]

print("False positives in top 50:", len(false_positives))

display(
    false_positives[
        [GROUP] + FEATS + ["score"]
    ].head(5)
)

# Missed positive examples
missed = te[te[LABEL] == 1].sort_values(
    "score",
    ascending=True
).head(5)

print("Missed positive examples:")

display(
    missed[[GROUP] + FEATS + ["score"]]
)


False positives in top 50: 18


,client_hash_id,impressions_log,ctr_early,avg_position_early,days_since_update,search_volume_log,impr_trend_log,score
690,client_73cda7b4e4f265ea,5.942799,0.015789,5.408948,23.0,3.044522,-0.744644,0.825168
1008,client_a80fca3f171ed1de,8.440744,0.001512,9.177116,25.0,0.000000,0.848223,0.775129
522,client_73cda7b4e4f265ea,8.514189,0.004213,4.229356,23.0,0.000000,-0.759788,0.770316
54,client_73cda7b4e4f265ea,8.251142,0.002871,5.196005,23.0,0.000000,-1.792977,0.768848
2433,client_73cda7b4e4f265ea,8.826441,0.003964,2.114805,23.0,0.000000,-0.369022,0.766431


Missed positive examples:


,client_hash_id,impressions_log,ctr_early,avg_position_early,days_since_update,search_volume_log,impr_trend_log,score
708,client_73cda7b4e4f265ea,7.407924,0.006675,3.784690,23.0,2.397895,0.410518,0.418499
2106,client_73cda7b4e4f265ea,7.202661,0.005961,4.355291,23.0,3.044522,0.345663,0.421346
665,client_73cda7b4e4f265ea,7.195937,0.018005,1.807467,23.0,2.397895,0.606715,0.446235
2049,client_73cda7b4e4f265ea,7.840313,0.006299,4.076035,23.0,2.397895,0.304454,0.446527
756,client_73cda7b4e4f265ea,7.250636,0.012784,2.513624,23.0,2.397895,0.242313,0.448225


In [ ]:
FEATS_WITHOUT_CTR = [
    f for f in FEATS if f != "ctr_early"
]

model_without_ctr = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=20,
        random_state=0,
        n_jobs=-1
    )
)

model_without_ctr.fit(
    tr_g[FEATS_WITHOUT_CTR],
    tr_g[LABEL]
)

scores_without_ctr = model_without_ctr.predict_proba(
    te_g[FEATS_WITHOUT_CTR]
)[:, 1]

print("Original grouped results:", after)
print(
    "Without ctr_early:",
    round(precision_at_k(te_g[LABEL], scores_without_ctr, 50), 3),
    round(roc_auc_score(te_g[LABEL], scores_without_ctr), 3)
)


Original grouped results: (0.64, np.float64(0.593))
Without ctr_early: 0.68 0.6


In [ ]:

print("Random split base rate:", round(te_a[LABEL].mean(), 3))
print("Grouped split base rate:", round(te_g[LABEL].mean(), 3))
print("Overall base rate:", round(d[LABEL].mean(), 3))


Random split base rate: 0.595
Grouped split base rate: 0.536
Overall base rate: 0.59


### Feature Timing Audit

The feature window covers March 1–20, 2026, while the target label uses clicks from March 21–31, 2026.

| Feature | Time Window | Audit Observation |
|---|---|---|
| impressions_log | March 1–20 | Early impressions |
| ctr_early | March 1–20 | Early clicks and impressions |
| avg_position_early | March 1–20 | Early average position |
| impr_trend_log | March 1–20 | Comparison of two early periods |
| days_since_update | As of March 20 | Content update date |
| search_volume_log | Unknown | Exact measurement timestamp is not established |

No late-period clicks are directly included in the feature calculations.

However, `ctr_early` shares `clicks_early` with the target formula. This creates a dependency between the feature and label construction that deserves further investigation.

The timing of `search_volume` should also be confirmed.

**Verdict:** No direct use of future-period clicks was identified in the listed features. However, the shared early-click dependency and unclear search-volume timestamp remain limitations.

In [ ]:
print("Random split:", before)
print("Grouped split:", after)

Random split: (0.74, np.float64(0.614))
Grouped split: (0.64, np.float64(0.593))


In [ ]:
train_clients = set(tr_g[GROUP].unique())
test_clients = set(te_g[GROUP].unique())

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Overlapping clients:", len(train_clients & test_clients))

print("\nLargest client share in grouped test set:")
print(round(te_g[GROUP].value_counts(normalize=True).iloc[0], 3))

print("\nGrouped test rows:", len(te_g))

Train clients: 15
Test clients: 7
Overlapping clients: 0

Largest client share in grouped test set:
0.966

Grouped test rows: 1400


In [ ]:
client_distribution = (
    te_g[GROUP]
    .value_counts()
    .reset_index(drop=True)
    .rename("rows")
)

print("Number of test clients:", len(client_distribution))
print("Largest client rows:", client_distribution.iloc[0])
print("Largest client share:", round(
    client_distribution.iloc[0] / len(te_g), 3
))
print("Smallest client rows:", client_distribution.iloc[-1])
print("Median rows per client:", client_distribution.median())

Number of test clients: 7
Largest client rows: 1353
Largest client share: 0.966
Smallest client rows: 2
Median rows per client: 10.0


In [ ]:

# Show error examples without client identifiers

error_features = FEATS + ["score"]

false_positives = (
    top[top[LABEL] == 0][error_features]
    .head(5)
)

missed = (
    te[te[LABEL] == 1]
    .sort_values("score", ascending=True)[error_features]
    .head(5)
)

print("False positives:")
display(false_positives.round(3))

print("Missed positives:")
display(missed.round(3))


False positives:


,impressions_log,ctr_early,avg_position_early,days_since_update,search_volume_log,impr_trend_log,score
690,5.943,0.016,5.409,23.0,3.045,-0.745,0.825
1008,8.441,0.002,9.177,25.0,0.000,0.848,0.775
522,8.514,0.004,4.229,23.0,0.000,-0.760,0.770
54,8.251,0.003,5.196,23.0,0.000,-1.793,0.769
2433,8.826,0.004,2.115,23.0,0.000,-0.369,0.766


Missed positives:


,impressions_log,ctr_early,avg_position_early,days_since_update,search_volume_log,impr_trend_log,score
708,7.408,0.007,3.785,23.0,2.398,0.411,0.418
2106,7.203,0.006,4.355,23.0,3.045,0.346,0.421
665,7.196,0.018,1.807,23.0,2.398,0.607,0.446
2049,7.840,0.006,4.076,23.0,2.398,0.304,0.447
756,7.251,0.013,2.514,23.0,2.398,0.242,0.448


## 4. Failure Analysis

I inspected false positives among the model's top 50 predictions and positive examples that received relatively low prediction scores.

- **False positives:** 18 of the top 50 predictions did not have the proxy decline label.
- **Missed positives:** Some records with the proxy decline label received relatively low prediction scores.

These examples show that the model's ranking is imperfect. A high prediction score does not guarantee the observed outcome, and the model does not identify every positive case.

The examples are descriptive and do not establish general failure patterns across all clients.

### Limitations

1. The target is a proxy based on click decline, not a verified judgment that content needs refreshing.
2. Random splitting may overestimate performance when records from the same clients appear in both sets.
3. The grouped test set is highly imbalanced, with one client contributing 96.6% of its rows.
4. The shared use of early clicks in `ctr_early` and the target formula requires further investigation.
5. The measurement timestamp for `search_volume` is unclear.
6. The evaluation covers one monthly dataset and does not establish performance across other months or a broad range of clients.

## 5. Claim Rewrite

| Original Claim | Problem | Safer Version |
|---|---|---|
| "The model predicts which pages need a refresh." | The label is a proxy, not proof of refresh need. | "The model ranks content records associated with the proxy decline label in this sample." |
| "Feature X drives performance." | Association does not establish causation. | "Feature X is associated with the target in this sample." |
| "The model works on any site." | Evaluation covers a limited and imbalanced client sample. | "The model was evaluated on 7 held-out clients, but one client dominated the test set." |
| "Removing CTR proves leakage." | Feature ablation does not prove leakage. | "Removing `ctr_early` improved both metrics in this grouped-split experiment." |

### Final Claim

In this March 2026 dataset, the model showed directional ranking ability for a proxy click-decline label. Performance decreased under grouped validation compared with random validation.

Because the grouped test set was highly imbalanced and the target is only a proxy, these results should be treated as exploratory decision-support evidence rather than proof that the model can reliably identify pages requiring a refresh.

## 5) Self-check

- [x] Two paper findings, each with a constructive methodology question
- [x] Before/after under an honest split, with the numbers shown
- [x] Leakage audit with a verdict per feature
- [x] Real error examples
- [x] Claims rewritten with observed / measured / directional / decision-support language
- [x] No client names, domains, URLs or tokens in any output